In [1]:
# ======================================
# 1. CONFIGURAÇÕES E PARÂMETROS GERAIS
# ======================================

import pandas as pd
import numpy as np
import random

qtd_fornecedores = 200
meses = 24
tx_anomalia_mes = 0.06
seed = 42

random.seed(seed)
np.random.seed(seed)

In [ ]:
# ======================================
# 2. GERACAO DE FORNECEDORES
# ======================================

gp_crescente = ['crescente'] * 80
gp_decrescente = ['decrescente'] * 60
gp_estavel = ['estavel'] * 60

grupos = gp_crescente + gp_decrescente + gp_estavel
random.shuffle(grupos)

# B2: papel comercial do parceiro. Antes o tipo da transacao era um cara-ou-coroa
# 60/40 independente do parceiro, o que produzia 8.029 RECEITAs vindas de
# FORNECEDOR e 5.646 DESPESAs indo para CLIENTE. Agora o papel e sorteado uma
# vez por parceiro e e ELE que determina o tipo da transacao (cel. 11).
papeis = ['FORNECEDOR'] * 90 + ['CLIENTE'] * 70 + ['AMBOS'] * 40
random.shuffle(papeis)

fornecedores = []

for i in range(200):
    opcao = grupos[i]
    padroes = ['NUM', 'ACC', 'FORN']

    padrao_atual = random.choice(padroes)

    if padrao_atual == 'NUM':
        id_fornecedor = f'{1000000+i}'

    elif padrao_atual == 'ACC':
        id_fornecedor = f'ACC-{i+1:05d}'

    elif padrao_atual == 'FORN':
        id_fornecedor = f'FORN-{i+1:05d}'

    valor_medio_inicial = round(np.random.lognormal(8, 0.6), 2)

    if valor_medio_inicial <= 4000:
        freq_media_inicial = np.random.randint(10, 19)
    elif valor_medio_inicial > 4000 and valor_medio_inicial <= 10000:
        freq_media_inicial = np.random.randint(6, 13)
    elif valor_medio_inicial > 10000:
        freq_media_inicial = np.random.randint(3, 9)

    prob_inicio = np.random.uniform(0, 1)

    if prob_inicio < 0.6:
        mes_inicio = np.random.randint(1, 4)
    else:
        mes_inicio = np.random.randint(4, 24)

    if mes_inicio > 18:
        mes_fim = 24
    else:
        prob_fim = np.random.uniform(0, 1)
        if prob_fim < 0.6:
            mes_fim = 24
        else:
            mes_fim = np.random.randint(mes_inicio + 6, 25)

    fornecedores.append({
        'id_fornecedor_raw': id_fornecedor,
        'papel': papeis[i],
        'grupo': opcao,
        'valor_medio_inicial': valor_medio_inicial,
        'freq_media_inicial': freq_media_inicial,
        'mes_inicio': mes_inicio,
        'mes_fim': mes_fim
    })
df_fornecedores = pd.DataFrame(fornecedores)

print('Parceiros por papel comercial:')
print(df_fornecedores['papel'].value_counts().to_string())


In [3]:
# ======================================
# 3. ESTRUTURA TEMPORAL (FORNECEDOR x MÊS)
# ======================================

df_meses = pd.DataFrame({'mes_pagamento': pd.RangeIndex(start=1, stop=25, step=1)})

df_fornecedor_mes_completo = df_fornecedores.merge(df_meses, how='cross')

df_fornecedor_mes_ativo = df_fornecedor_mes_completo.loc[
    (df_fornecedor_mes_completo['mes_pagamento']>=df_fornecedor_mes_completo['mes_inicio']) & 
    (df_fornecedor_mes_completo['mes_pagamento']<=df_fornecedor_mes_completo['mes_fim'])
]

display(df_fornecedor_mes_ativo)

,id_fornecedor_raw,grupo,valor_medio_inicial,freq_media_inicial,mes_inicio,mes_fim,mes_pagamento
9,1000000,crescente,4015.95,8,10,24,10
10,1000000,crescente,4015.95,8,10,24,11
11,1000000,crescente,4015.95,8,10,24,12
12,1000000,crescente,4015.95,8,10,24,13
13,1000000,crescente,4015.95,8,10,24,14
...,...,...,...,...,...,...,...
4790,1000199,estavel,2483.11,13,3,19,15
4791,1000199,estavel,2483.11,13,3,19,16
4792,1000199,estavel,2483.11,13,3,19,17
4793,1000199,estavel,2483.11,13,3,19,18


In [4]:
# ======================================
# 4. MODELAGEM DE FREQUÊNCIA (POISSON)
# ======================================

taxa_freq = 0.015

min_lambda = 1.0
df_fornecedor_mes_ativo['mes_relativo'] = df_fornecedor_mes_ativo['mes_pagamento'] - df_fornecedor_mes_ativo['mes_inicio']
df_fornecedor_mes_ativo['lambda_mes'] = df_fornecedor_mes_ativo['freq_media_inicial']
#print(df_fornecedor_mes_ativo)

mask_crescente = df_fornecedor_mes_ativo['grupo'] == 'crescente'
#display(df_fornecedor_mes_ativo)
#display(mask_crescente)
df_fornecedor_mes_ativo['lambda_mes'] = df_fornecedor_mes_ativo.lambda_mes.astype(float)

# Crescente
df_fornecedor_mes_ativo.loc[ 
    mask_crescente, 
    'lambda_mes' 
    ] = df_fornecedor_mes_ativo['freq_media_inicial'] * (1 + taxa_freq) ** df_fornecedor_mes_ativo['mes_relativo']


mask_decrescente = df_fornecedor_mes_ativo['grupo'] == 'decrescente'
# Decrescente
df_fornecedor_mes_ativo.loc[ 
    mask_decrescente, 
    'lambda_mes' 
    ] = (df_fornecedor_mes_ativo['freq_media_inicial'] * (1 - taxa_freq) ** df_fornecedor_mes_ativo['mes_relativo']).clip(lower=min_lambda)

# display(df_fornecedor_mes_ativo[df_fornecedor_mes_ativo['id_fornecedor_raw'] == 'FORN-00003'])



C:\Users\Adam\AppData\Local\Temp\ipykernel_21956\578324769.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fornecedor_mes_ativo['mes_relativo'] = df_fornecedor_mes_ativo['mes_pagamento'] - df_fornecedor_mes_ativo['mes_inicio']
C:\Users\Adam\AppData\Local\Temp\ipykernel_21956\578324769.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fornecedor_mes_ativo['lambda_mes'] = df_fornecedor_mes_ativo['freq_media_inicial']
C:\Users\Adam\AppData\Local\Temp\ipykernel_21956\578324769.py:15: SettingWithCopy

In [5]:
# ======================================
# 5. MODELAGEM DE VALOR MÉDIO MENSAL
# ======================================

taxa_valor = 0.005

df_fornecedor_mes_ativo["quantidade_real_pagamentos"] = np.random.poisson(df_fornecedor_mes_ativo["lambda_mes"])

# Crescente
df_fornecedor_mes_ativo.loc[ 
    mask_crescente, 
    'valor_medio_mes' 
    ] = df_fornecedor_mes_ativo['valor_medio_inicial'] * (1 + taxa_valor) ** df_fornecedor_mes_ativo['mes_relativo']

# Decrescente
df_fornecedor_mes_ativo.loc[ 
    mask_decrescente, 
    'valor_medio_mes' 
    ] = df_fornecedor_mes_ativo['valor_medio_inicial'] * (1 - taxa_valor) ** df_fornecedor_mes_ativo['mes_relativo']

# Estável
mask_estavel = df_fornecedor_mes_ativo['grupo'] == 'estavel'
df_fornecedor_mes_ativo.loc[ 
    mask_estavel, 
    'valor_medio_mes' 
    ] = df_fornecedor_mes_ativo['valor_medio_inicial']

display(df_fornecedor_mes_ativo)
#display(df_fornecedor_mes_ativo)

C:\Users\Adam\AppData\Local\Temp\ipykernel_21956\3764866049.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fornecedor_mes_ativo["quantidade_real_pagamentos"] = np.random.poisson(df_fornecedor_mes_ativo["lambda_mes"])
C:\Users\Adam\AppData\Local\Temp\ipykernel_21956\3764866049.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fornecedor_mes_ativo.loc[


,id_fornecedor_raw,grupo,valor_medio_inicial,freq_media_inicial,mes_inicio,mes_fim,mes_pagamento,mes_relativo,lambda_mes,quantidade_real_pagamentos,valor_medio_mes
9,1000000,crescente,4015.95,8,10,24,10,0,8.000000,7,4015.950000
10,1000000,crescente,4015.95,8,10,24,11,1,8.120000,8,4036.029750
11,1000000,crescente,4015.95,8,10,24,12,2,8.241800,13,4056.209899
12,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948
13,1000000,crescente,4015.95,8,10,24,14,4,8.490908,12,4096.873403
...,...,...,...,...,...,...,...,...,...,...,...
4790,1000199,estavel,2483.11,13,3,19,15,12,13.000000,10,2483.110000
4791,1000199,estavel,2483.11,13,3,19,16,13,13.000000,22,2483.110000
4792,1000199,estavel,2483.11,13,3,19,17,14,13.000000,8,2483.110000
4793,1000199,estavel,2483.11,13,3,19,18,15,13.000000,7,2483.110000


In [6]:
# ======================================
# 6. EXPLOSÃO TRANSACIONAL
# ======================================

sigma_valor = 0.10

df_pgtos = df_fornecedor_mes_ativo.loc[df_fornecedor_mes_ativo.index.repeat(df_fornecedor_mes_ativo['quantidade_real_pagamentos'])].reset_index(drop=True)

display(df_pgtos[(df_pgtos['id_fornecedor_raw'] == '1000000') & (df_pgtos['mes_pagamento'] == 13)])

,id_fornecedor_raw,grupo,valor_medio_inicial,freq_media_inicial,mes_inicio,mes_fim,mes_pagamento,mes_relativo,lambda_mes,quantidade_real_pagamentos,valor_medio_mes
28,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948
29,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948
30,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948
31,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948
32,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948
33,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948
34,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948
35,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948
36,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948


In [7]:
# ======================================
# 7. MODELAGEM MONETÁRIA
# ======================================

proporcao_ajuste = 0.3
desconto_min = 0.01 
desconto_max = 0.05 

N = len(df_pgtos)
array =  np.random.uniform(0, 1, N) # [1, 2, 3, 4, 5]
#print(array)
flag = array < proporcao_ajuste
#print(flag)
df_pgtos["flag_ajuste"] = flag

#df_pgtos[(df_pgtos['id_fornecedor_raw'] == '1000000') & (df_pgtos['mes_relativo'] == 2)]

# for i in range(len(df_pgtos)): 
#     prob_ajuste = np.random.uniform(0, 1) 
#     if prob_ajuste < 0.3: 
#         df_pgtos.loc[i, "flag_ajuste"] = True 
#     else: 
#         df_pgtos.loc[i, "flag_ajuste"] = False 

# #display(df_pgtos[(df_pgtos['flag_ajuste'] == True)]) 

# display(df_pgtos)

df_pgtos["mu_valor"] = np.log(df_pgtos['valor_medio_mes']) - (sigma_valor**2)/2

df_pgtos["valor_bruto"] = np.random.lognormal(df_pgtos["mu_valor"], sigma_valor)

display(df_pgtos)



df_pgtos["desconto"] = 0.0
mask_disk = df_pgtos['flag_ajuste'] == True

N = len(mask_disk)
desconto_min = 0.01 
desconto_max = 0.05 
desconto = np.random.uniform(desconto_min, desconto_max, N)

df_pgtos.loc[ 
    mask_disk, 
    'desconto' 
    ] = desconto[mask_disk]

df_pgtos['desconto'] = df_pgtos['valor_bruto'] * df_pgtos['desconto']

df_pgtos['valor_liquido'] = df_pgtos['valor_bruto'] - df_pgtos['desconto']
# df_pgtos[(df_pgtos['flag_ajuste'] == True)]

# df_pgtos[(df_pgtos['flag_ajuste'] == False) & (df_pgtos['desconto'] > 0)]

df_pgtos["id_transacao_raw"] = [f'TR-{i:05d}' for i in range(1, len(df_pgtos) + 1)]

df_pgtos

,id_fornecedor_raw,grupo,valor_medio_inicial,freq_media_inicial,mes_inicio,mes_fim,mes_pagamento,mes_relativo,lambda_mes,quantidade_real_pagamentos,valor_medio_mes,flag_ajuste,mu_valor,valor_bruto
0,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,3966.097441
1,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,4475.940332
2,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,4101.466781
3,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,4421.324444
4,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,3718.020715
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39787,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,2483.11,False,7.812267,1976.352737
39788,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,2483.11,False,7.812267,2192.120003
39789,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,2483.11,False,7.812267,3005.351230
39790,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,2483.11,False,7.812267,2185.524770


,id_fornecedor_raw,grupo,valor_medio_inicial,freq_media_inicial,mes_inicio,mes_fim,mes_pagamento,mes_relativo,lambda_mes,quantidade_real_pagamentos,valor_medio_mes,flag_ajuste,mu_valor,valor_bruto,desconto,valor_liquido,id_transacao_raw
0,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,3966.097441,0.000000,3966.097441,TR-00001
1,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,4475.940332,0.000000,4475.940332,TR-00002
2,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,4101.466781,0.000000,4101.466781,TR-00003
3,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,4421.324444,0.000000,4421.324444,TR-00004
4,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,3718.020715,0.000000,3718.020715,TR-00005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39787,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,2483.11,False,7.812267,1976.352737,0.000000,1976.352737,TR-39788
39788,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,2483.11,False,7.812267,2192.120003,0.000000,2192.120003,TR-39789
39789,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,2483.11,False,7.812267,3005.351230,0.000000,3005.351230,TR-39790
39790,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,2483.11,False,7.812267,2185.524770,0.000000,2185.524770,TR-39791


In [ ]:
# ======================================
# 8. MODELAGEM TEMPORAL TRANSACIONAL
# ======================================

data_base = pd.to_datetime('2024-01-01')

df_pgtos['data'] = data_base + df_pgtos['mes_pagamento'].apply(lambda x: pd.DateOffset(months=x-1))

df_pgtos['dias_no_mes'] = pd.DatetimeIndex(df_pgtos['data']).days_in_month

triangulo = np.random.triangular(left=0, mode=0.5, right=1, size=len(df_pgtos))

df_pgtos['dia_pgto'] = triangulo * (df_pgtos['dias_no_mes'])

df_pgtos['dia_pgto'] = df_pgtos.dia_pgto.astype(int)

# Off-by-one da origem: triangulo pertence a [0,1), entao dia_pgto podia dar 0 e
# 'data + (0-1) dias' caia no ULTIMO DIA DO MES ANTERIOR. O codigo original nao
# percebia porque derivava data_competencia da propria data_transacao depois.
# Fixando o dia no intervalo valido [1, dias_no_mes].
df_pgtos['dia_pgto'] = df_pgtos['dia_pgto'].clip(lower=1, upper=df_pgtos['dias_no_mes'])

df_pgtos['data_transacao'] = df_pgtos['data'] + pd.to_timedelta(df_pgtos['dia_pgto'] - 1, unit='D')

df_pgtos['data_transacao'] = pd.to_datetime(
    df_pgtos['data_transacao'],
    errors='coerce'
)

# B4: fim de semana empurrado para o dia util MAIS PROXIMO.
# Antes: sabado +2 e domingo +1, ambos caindo na segunda -- o que empilhava
# 17.049 das 39.792 transacoes (42,8%) numa segunda-feira. Era artefato do
# gerador, e um modelo que usasse dia da semana como feature aprenderia o
# artefato em vez do negocio.
# Agora: sabado -> sexta (-1), domingo -> segunda (+1).
_dow = df_pgtos['data_transacao'].dt.dayofweek
mask_sabado = _dow == 5
mask_domingo = _dow == 6

df_pgtos.loc[mask_sabado, 'data_transacao'] -= pd.Timedelta(days=1)
df_pgtos.loc[mask_domingo, 'data_transacao'] += pd.Timedelta(days=1)

# Guarda de borda: o deslocamento nao pode jogar a transacao para outro mes
# (mudaria a competencia). Nesses casos vai para o dia util do outro lado.
_mes_alvo = pd.DatetimeIndex(df_pgtos['data']).month
_vazou_sab = mask_sabado & (df_pgtos['data_transacao'].dt.month != _mes_alvo)
df_pgtos.loc[_vazou_sab, 'data_transacao'] += pd.Timedelta(days=3)   # -> segunda

_vazou_dom = mask_domingo & (df_pgtos['data_transacao'].dt.month != _mes_alvo)
df_pgtos.loc[_vazou_dom, 'data_transacao'] -= pd.Timedelta(days=3)   # -> sexta

assert (df_pgtos['data_transacao'].dt.dayofweek < 5).all(), 'sobrou transacao em fim de semana'
assert (df_pgtos['data_transacao'].dt.month == _mes_alvo).all(), 'transacao vazou de competencia'

print('Distribuicao por dia da semana (0=seg ... 4=sex):')
print(df_pgtos['data_transacao'].dt.dayofweek.value_counts().sort_index().to_string())

df_pgtos['data_competencia'] = df_pgtos['data_transacao'].dt.to_period('M')


In [9]:
# ======================================
# 9. SEGMENTAÇÃO DE PERFIL DE RISCO
# ======================================

P90 = df_pgtos['valor_bruto'].quantile(0.9)
df_pgtos['flag_alto_valor'] = df_pgtos['valor_bruto'] > P90

df_pgtos

# df_pgtos[df_pgtos['P90'] == True]

# perfil_risco

condicoes = [
    (df_pgtos['flag_ajuste'] == True) & (df_pgtos['flag_alto_valor'] == True) ,
    (df_pgtos['flag_ajuste'] == True) & (df_pgtos['flag_alto_valor'] == False),
    (df_pgtos['flag_ajuste'] == False) & (df_pgtos['flag_alto_valor'] == True),
    (df_pgtos['flag_ajuste'] == False) & (df_pgtos['flag_alto_valor'] == False)]

escolhas = ['alto_valor_retencao', 'retencao', 'alto_valor', 'normal']

df_pgtos['perfil_risco'] = np.select(condicoes, escolhas, default='Desconecido')

alto_valor = df_pgtos[df_pgtos['perfil_risco'] == 'alto_valor_retencao']['perfil_risco'].count()

In [10]:
# ======================================
# 10. VALIDAÇÕES ESTATÍSTICAS
# ======================================

display(df_pgtos)

,id_fornecedor_raw,grupo,valor_medio_inicial,freq_media_inicial,mes_inicio,mes_fim,mes_pagamento,mes_relativo,lambda_mes,quantidade_real_pagamentos,...,desconto,valor_liquido,id_transacao_raw,data,dias_no_mes,dia_pgto,data_transacao,data_competencia,flag_alto_valor,perfil_risco
0,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,0.000000,3966.097441,TR-00001,2024-10-01 00:00:00,31,14,2024-10-14,2024-10,False,normal
1,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,0.000000,4475.940332,TR-00002,2024-10-01 00:00:00,31,9,2024-10-09,2024-10,False,normal
2,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,0.000000,4101.466781,TR-00003,2024-10-01 00:00:00,31,9,2024-10-09,2024-10,False,normal
3,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,0.000000,4421.324444,TR-00004,2024-10-01 00:00:00,31,14,2024-10-14,2024-10,False,normal
4,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,0.000000,3718.020715,TR-00005,2024-10-01 00:00:00,31,10,2024-10-10,2024-10,False,normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39787,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,0.000000,1976.352737,TR-39788,2025-07-01 00:00:00,31,22,2025-07-22,2025-07,False,normal
39788,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,0.000000,2192.120003,TR-39789,2025-07-01 00:00:00,31,21,2025-07-21,2025-07,False,normal
39789,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,0.000000,3005.351230,TR-39790,2025-07-01 00:00:00,31,18,2025-07-18,2025-07,False,normal
39790,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,0.000000,2185.524770,TR-39791,2025-07-01 00:00:00,31,26,2025-07-28,2025-07,False,normal


In [ ]:
# ======================================
# 11. DEFINICAO DO TIPO DE TRANSACAO
# ======================================

N = len(df_pgtos)

# B2: o tipo da transacao decorre do PAPEL do parceiro (definido uma vez por
# parceiro na cel. 2). Antes era um cara-ou-coroa 60/40 independente do
# parceiro, o que gerava 8.029 RECEITAs vindas de FORNECEDOR e 5.646 DESPESAs
# indo para CLIENTE -- incoerencia que contaminava qualquer analise por parceiro.
#   FORNECEDOR -> a empresa paga        => DESPESA
#   CLIENTE    -> a empresa recebe      => RECEITA
#   AMBOS      -> os dois fluxos (60/40)
papel_arr = df_pgtos['papel'].to_numpy()
sorteio_ambos = np.random.uniform(0, 1, N)

df_pgtos['tipo_transacao'] = np.where(
    papel_arr == 'FORNECEDOR', 'DESPESA',
    np.where(
        papel_arr == 'CLIENTE', 'RECEITA',
        np.where(sorteio_ambos < 0.6, 'DESPESA', 'RECEITA')
    )
)

print('Tipo de transacao por papel do parceiro:')
print(pd.crosstab(df_pgtos['papel'], df_pgtos['tipo_transacao']))

df_pgtos['status_pagamento'] = None

mask_despesa_normal = (df_pgtos['tipo_transacao'] == 'DESPESA') & (df_pgtos['perfil_risco'] == 'normal')
mask_despesa_retencao = (df_pgtos['tipo_transacao'] == 'DESPESA') & (df_pgtos['perfil_risco'] == 'retencao')
mask_despesa_alto_valor = (df_pgtos['tipo_transacao'] == 'DESPESA') & (df_pgtos['perfil_risco'] == 'alto_valor')
mask_despesa_alto_valor_retencao = (df_pgtos['tipo_transacao'] == 'DESPESA') & (df_pgtos['perfil_risco'] == 'alto_valor_retencao')

mask_receita_normal = (df_pgtos['tipo_transacao'] == 'RECEITA') & (df_pgtos['perfil_risco'] == 'normal')
mask_receita_retencao = (df_pgtos['tipo_transacao'] == 'RECEITA') & (df_pgtos['perfil_risco'] == 'retencao')
mask_receita_alto_valor = (df_pgtos['tipo_transacao'] == 'RECEITA') & (df_pgtos['perfil_risco'] == 'alto_valor')
mask_receita_alto_valor_retencao = (df_pgtos['tipo_transacao'] == 'RECEITA') & (df_pgtos['perfil_risco'] == 'alto_valor_retencao')

def aplicar_status(mask, p_pago, p_atrasado, p_cancelado):
    N = mask.sum()

    if N == 0:
        return

    sorteios = np.random.uniform(0, 1, N)

    limite_pago = p_pago
    limite_atrasado = p_pago + p_atrasado

    status_grupo = np.empty(N, dtype=object)

    for i, sorteio in enumerate(sorteios):
        if sorteio < limite_pago:
            status_grupo[i] = "PAGO"
        elif sorteio < limite_atrasado:
            status_grupo[i] = "ATRASADO"
        else:
            status_grupo[i] = "CANCELADO"

    df_pgtos.loc[mask, 'status_pagamento'] = status_grupo

aplicar_status(mask_despesa_normal, 0.85, 0.10, 0.05)
aplicar_status(mask_despesa_retencao, 0.72, 0.20, 0.08)
aplicar_status(mask_despesa_alto_valor, 0.92, 0.06, 0.02)
aplicar_status(mask_despesa_alto_valor_retencao, 0.80, 0.14, 0.06)
aplicar_status(mask_receita_normal, 0.80, 0.14, 0.06)
aplicar_status(mask_receita_retencao, 0.68, 0.24, 0.08)
aplicar_status(mask_receita_alto_valor, 0.88, 0.09, 0.03)
aplicar_status(mask_receita_alto_valor_retencao, 0.75, 0.17, 0.08)

assert df_pgtos['status_pagamento'].isnull().sum() == 0, 'transacao sem status_pagamento'

df_base_normal = df_pgtos.copy()


In [12]:
# 12. INJEÇÃO DE ANOMALIA DE VALOR

anomalias_por_fornecedor = []

df_contagem_fornecedores = df_pgtos.groupby('id_fornecedor_raw').size().reset_index(name='qtd_transacoes')

df_contagem_fornecedores

for fornecedor in df_contagem_fornecedores['id_fornecedor_raw']:
    
    subset_fornecedor = df_pgtos[df_pgtos['id_fornecedor_raw'] == fornecedor]

    qtd_transacoes = len(subset_fornecedor)

    qtd_anomalias = int(np.ceil(qtd_transacoes * tx_anomalia_mes))

    if qtd_anomalias == 0:
        continue

    anomalias_por_fornecedor.append({
        'id_fornecedor_raw': fornecedor,
        'qtd_transacoes': qtd_transacoes,
        'qtd_anomalias': qtd_anomalias,
        'percentual': (qtd_anomalias / qtd_transacoes) * 100
    })

df_anomalias_config = pd.DataFrame(anomalias_por_fornecedor)

df_anomalias_config

,id_fornecedor_raw,qtd_transacoes,qtd_anomalias,percentual
0,1000000,139,9,6.474820
1,1000004,198,12,6.060606
2,1000014,147,9,6.122449
3,1000015,123,8,6.504065
4,1000016,147,9,6.122449
...,...,...,...,...
195,FORN-00192,111,7,6.306306
196,FORN-00193,162,10,6.172840
197,FORN-00195,427,26,6.088993
198,FORN-00196,173,11,6.358382


In [13]:
for_ids = []

map_valor = df_pgtos.set_index('id_transacao_raw')['valor_bruto'].to_dict()

for fornecedor in df_anomalias_config['id_fornecedor_raw']:
    subset_fornecedor = df_anomalias_config[df_anomalias_config['id_fornecedor_raw'] == fornecedor]
    
    fornecedor = subset_fornecedor['id_fornecedor_raw'].values[0]
    qtd_anomalias = int(subset_fornecedor['qtd_anomalias'].values[0])
    id_transacao_raw = df_pgtos[df_pgtos['id_fornecedor_raw'] == fornecedor]['id_transacao_raw'].tolist()

    qtd_anomalias = min(qtd_anomalias, len(id_transacao_raw))

    if qtd_anomalias <= 0 or len(id_transacao_raw) == 0:
        continue

    ids = np.random.choice(id_transacao_raw, size=qtd_anomalias, replace=False)
    
    ids_lista = ids.tolist()

    for id_sorteado in ids:
        if np.random.uniform(0, 1) < 0.7:
            fator_aplicado = np.random.uniform(1.5, 3.5)  # Aumento entre 50% e 250%
        else: 
            fator_aplicado = np.random.uniform(0.2, 0.6)  # Redução entre 20% e 60%

        for_ids.append({
            'id_fornecedor_raw': fornecedor,
            'qtd_anomalias': qtd_anomalias,
            'id_transacao_raw': id_sorteado, 
            'fator_aplicado': float(fator_aplicado),
            'valor_original': map_valor[id_sorteado],
            'valor_novo': map_valor[id_sorteado] * fator_aplicado
        })

df_anomalias_valor = pd.DataFrame(for_ids)

In [ ]:
df_base_anomalo = df_base_normal.copy()

df_base_anomalo['flag_anomalia'] = 0

# merge anomalies; avoid duplicate fornecedor columns

df_base_anomalo = df_base_anomalo.merge(
    df_anomalias_valor,
    on='id_transacao_raw',
    how='left'
)

# after merge pandas appends _x/_y suffixes for shared column names
# keep the original left-side id_fornecedor_raw and drop any extra
if 'id_fornecedor_raw_y' in df_base_anomalo.columns:
    df_base_anomalo.drop(columns=['id_fornecedor_raw_y'], inplace=True)
if 'id_fornecedor_raw_x' in df_base_anomalo.columns:
    df_base_anomalo.rename(columns={'id_fornecedor_raw_x': 'id_fornecedor_raw'}, inplace=True)

# Anomala = linha que recebeu fator na injecao (cel. 12).
# notna() e explicito: antes usava 'fator_aplicado >= 0', que so funcionava
# porque NaN >= 0 avalia False por acaso.
df_base_anomalo['flag_anomalia'] = df_base_anomalo['fator_aplicado'].notna()

mask_anomalo = df_base_anomalo['flag_anomalia'] == True

df_base_anomalo.loc[
    mask_anomalo,
    'valor_bruto'
] = df_base_anomalo.loc[mask_anomalo, 'valor_novo']

df_base_anomalo['valor_liquido'] = df_base_anomalo['valor_bruto'] - df_base_anomalo['desconto']

# A9: valores monetarios arredondados na origem. O raw gravava
# 2616.110124681177 num campo que a camada trusted declara como DECIMAL(18,2) --
# o Postgres arredondaria em silencio na carga. Arredondar aqui alinha as
# camadas e elimina a instabilidade de ultimo bit que desestabilizava o
# raw_row_hash (ver A8).
df_base_anomalo['valor_bruto'] = df_base_anomalo['valor_bruto'].round(2)
df_base_anomalo['valor_liquido'] = df_base_anomalo['valor_liquido'].round(2)
df_base_anomalo['valor_original'] = df_base_anomalo['valor_original'].round(2)

assert (df_base_anomalo['valor_liquido'] <= df_base_anomalo['valor_bruto']).all(), \
    'valor_liquido maior que valor_bruto'
assert (df_base_anomalo['valor_bruto'] > 0).all(), 'valor_bruto nao positivo'


In [ ]:
# C3: a coluna se chama codigo_area_raw porque e isso que ela guarda --
# o CODIGO da area ('FIN', 'TI'), nao o id numerico de raw.areas.
# Antes se chamava id_area_raw, o que fazia o join com raw.areas parecer
# errado: t.id_area_raw = a.codigo_area.
# Áreas

# IDs de área — inclui todos os 20 códigos da dimensão raw.areas
# Os 8 originais mantêm dominância; 12 complementares com peso menor
id_areas = [
    'FIN', 'COMP', 'OP', 'COM', 'RH', 'TI', 'MKT', 'JUR',
    'ADM', 'CTB', 'LOG', 'CPT', 'PJT', 'QAS', 'ATD', 'VND', 'PES', 'PRD', 'SEC', 'ESG'
]

_pesos_raw_areas = [15, 18, 17, 14, 10, 9, 9, 8,   # originais (dominantes)
                    5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]  # complementares
peso = [p / sum(_pesos_raw_areas) for p in _pesos_raw_areas]

areas = [
    'Financeiro', 'Compliance', 'Operações', 'Comercial',
    'Recursos Humanos', 'TI', 'Marketing', 'Jurídico',
    'Administração', 'Contabilidade', 'Logística', 'Compras',
    'Projetos', 'Qualidade', 'Atendimento', 'Vendas',
    'Pesquisa', 'Produto', 'Segurança', 'Sustentabilidade'
]

N = len(df_base_anomalo)

array_areas = np.random.choice(id_areas, p=peso, size=N)

df_base_anomalo["codigo_area_raw"] = array_areas

df_base_anomalo

In [ ]:
# categoria
#
# B1 + B3: a categoria agora e COERENTE com o tipo da transacao e correlacionada
# com a area. Antes era um np.random.choice sobre os 40 ids, independente de
# tudo -- o que produzia 18.847 linhas (47%) com RECEITA classificada em
# categoria de DESPESA (ex.: "RECEITA / Salarios e Ordenados") e deixava
# dim_area sem nenhum poder analitico.
#
# Regra: sorteia dentro do conjunto do tipo certo; com 70% de chance dentro do
# subconjunto tipico da area, com 30% em qualquer categoria daquele tipo
# (representa o gasto eventual fora do padrao da area).

CATEGORIAS_RECEITA = [1, 2, 3, 31, 32, 33, 34, 35, 36, 37]
CATEGORIAS_DESPESA = list(range(4, 31)) + [38, 39, 40]

# Categorias tipicas de cada area. Areas ausentes usam o conjunto completo.
AFINIDADE_DESPESA = {
    'FIN':  [21, 22, 23],           # seguros, taxas bancarias, juros
    'COMP': [24, 25, 26],           # juridico, contabil, certificados
    'OP':   [16, 17, 28],           # combustivel, manut. veicular, equipamentos
    'COM':  [13, 18, 19, 20],       # publicidade, viagens
    'RH':   [4, 5, 30],             # salarios, encargos, treinamento
    'TI':   [27, 28, 29],           # software, equipamentos, suporte
    'MKT':  [13, 14, 15],           # publicidade, eventos, material
    'JUR':  [11, 24],               # honorarios, despesas juridicas
    'ADM':  [6, 7, 8, 9],           # escritorio, aluguel, energia, telefonia
    'CTB':  [22, 25, 40],           # taxas, assessoria contabil, impostos
    'LOG':  [16, 17],               # combustivel, manutencao veicular
    'CPT':  [6, 12],                # material, consultorias
    'PJT':  [11, 12],               # honorarios, consultorias
    'QAS':  [12, 30],               # consultorias, treinamento
    'ATD':  [9, 29],                # telefonia, suporte tecnico
    'VND':  [13, 38, 39],           # publicidade, descontos, devolucoes
    'PES':  [12, 30],               # consultorias, treinamento
    'PRD':  [12, 27],               # consultorias, software
    'SEC':  [10, 21],               # manutencao predial, seguros
    'ESG':  [12, 30],               # consultorias, treinamento
}
AFINIDADE_RECEITA = {
    'FIN':  [3, 34],                # receita financeira, equivalencia
    'COM':  [1, 2, 33],             # vendas, comissoes
    'VND':  [1, 2, 33],
    'MKT':  [33],                   # comissoes
    'ADM':  [31, 35],               # aluguel, venda de ativo
    'JUR':  [37],                   # receitas eventuais
    'CTB':  [3, 34],
    'OP':   [1, 2],
}

rng_cat = np.random.RandomState(seed + 5)

areas_arr = df_base_anomalo["codigo_area_raw"].to_numpy()
tipos_arr = df_base_anomalo["tipo_transacao"].to_numpy()
sorteio_afinidade = rng_cat.uniform(0, 1, len(df_base_anomalo))

categorias = np.empty(len(df_base_anomalo), dtype=int)
for i in range(len(df_base_anomalo)):
    if tipos_arr[i] == 'RECEITA':
        completo, afins = CATEGORIAS_RECEITA, AFINIDADE_RECEITA.get(areas_arr[i])
    else:
        completo, afins = CATEGORIAS_DESPESA, AFINIDADE_DESPESA.get(areas_arr[i])
    conjunto = afins if (afins and sorteio_afinidade[i] < 0.70) else completo
    categorias[i] = conjunto[rng_cat.randint(len(conjunto))]

df_base_anomalo["id_categoria_raw"] = categorias

print('Categorias por tipo de transacao (deve haver ZERO cruzamento):')
_chk = df_base_anomalo.assign(
    tipo_cat=np.where(df_base_anomalo.id_categoria_raw.isin(CATEGORIAS_RECEITA),
                      'RECEITA', 'DESPESA')
)
print(pd.crosstab(_chk.tipo_transacao, _chk.tipo_cat))


In [ ]:
# moeda
#
# D4: base fixada em BRL. Antes eram sorteadas BRL/USD/EUR com pesos
# 0.94/0.05/0.01, mas os valores em USD e EUR ficavam na MESMA escala do BRL e
# nao havia taxa_cambio em lugar nenhum -- somar as tres (o que qualquer
# dashboard faz) produzia numero sem significado. Alem disso 199 dos 200
# parceiros transacionavam nas 3 moedas aleatoriamente, entao nao havia padrao
# a preservar. O escopo passa a ser operacao nacional; trusted.moedas e
# refined.dim_moeda seguem populadas com BRL/USD/EUR para extensibilidade.

df_base_anomalo["moeda"] = "BRL"


In [18]:
# forma de pagamento

forma_pagamento = ['PIX', 'BOLETO', 'TED', 'TRANSFERENCIA', 'CARTAO']

peso_forma_pagamento = [0.28, 0.24, 0.20, 0.18, 0.10]

escolha_tp_pgto = np.random.choice(forma_pagamento, p=peso_forma_pagamento, size=N)

df_base_anomalo["forma_pagamento"] = escolha_tp_pgto

df_base_anomalo

,id_fornecedor_raw,grupo,valor_medio_inicial,freq_media_inicial,mes_inicio,mes_fim,mes_pagamento,mes_relativo,lambda_mes,quantidade_real_pagamentos,...,status_pagamento,flag_anomalia,qtd_anomalias,fator_aplicado,valor_original,valor_novo,id_area_raw,id_categoria_raw,moeda,forma_pagamento
0,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,PAGO,False,NaN,NaN,NaN,NaN,JUR,27,BRL,BOLETO
1,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,PAGO,False,NaN,NaN,NaN,NaN,COM,17,BRL,TED
2,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,PAGO,False,NaN,NaN,NaN,NaN,SEC,17,BRL,PIX
3,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,PAGO,False,NaN,NaN,NaN,NaN,ADM,4,BRL,PIX
4,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,PAGO,False,NaN,NaN,NaN,NaN,JUR,1,BRL,BOLETO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39787,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,PAGO,False,NaN,NaN,NaN,NaN,ESG,40,BRL,PIX
39788,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,PAGO,False,NaN,NaN,NaN,NaN,ATD,5,BRL,PIX
39789,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,PAGO,False,NaN,NaN,NaN,NaN,PJT,17,BRL,TRANSFERENCIA
39790,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,PAGO,False,NaN,NaN,NaN,NaN,TI,3,BRL,TED


In [ ]:
# Mapear id_categoria_raw para nome
map_categoria = {
    1: "Venda de Produtos",
    2: "Venda de Serviços",
    3: "Receita Financeira",
    4: "Salários e Ordenados",
    5: "Encargos Trabalhistas",
    6: "Material de Escritório",
    7: "Aluguel",
    8: "Energia Elétrica",
    9: "Telefonia e Internet",
    10: "Manutenção Predial",
    11: "Honorários Profissionais",
    12: "Consultorias",
    13: "Publicidade e Propaganda",
    14: "Eventos e Promoções",
    15: "Material de Marketing",
    16: "Combustível e Lubrificantes",
    17: "Manutenção Veicular",
    18: "Passagens Aéreas",
    19: "Hospedagem",
    20: "Alimentação em Viagem",
    21: "Seguros",
    22: "Taxas Bancárias",
    23: "Juros e Encargos",
    24: "Despesas Jurídicas",
    25: "Assessoria Contábil",
    26: "Certificados Digitais",
    27: "Software e Licenças",
    28: "Manutenção de Equipamentos",
    29: "Suporte Técnico",
    30: "Treinamento e Capacitação",
    31: "Receita de Aluguel",
    32: "Royalties",
    33: "Comissões Recebidas",
    34: "Resultado de Equivalência",
    35: "Venda de Ativo Imobilizado",
    36: "Doações Recebidas",
    37: "Receitas Eventuais",
    38: "Descontos Concedidos",
    39: "Devoluções de Vendas",
    40: "Impostos sobre Vendas"
}

# Templates por categoria
templates = {
    1: [
        "Prestação de serviço para área {area}",
        "Contrato de serviço - setor {area}",
        "Serviço especializado solicitado por {area}"
    ],
    2: [
        "Aquisição de material operacional - {area}",
        "Compra de insumos para setor {area}"
    ],
    3: [
        "Despesa de infraestrutura - {area}",
        "Manutenção estrutural vinculada à área {area}"
    ],
    4: [
        "Campanha institucional - {area}",
        "Investimento em marketing corporativo ({area})"
    ],
    5: [
        "Licenciamento de software - {area}",
        "Serviço de tecnologia contratado por {area}"
    ],
    6: [
        "Despesa de RH - {area}",
        "Serviço relacionado a recursos humanos ({area})"
    ],
    7: [
        "Honorários jurídicos - {area}",
        "Assessoria legal vinculada à área {area}"
    ],
    8: [
        "Despesa operacional - {area}",
        "Custo operacional associado a {area}"
    ],
    9: [
        "Serviço de telefonia e internet - {area}",
        "Contrato de conectividade para área {area}"
    ],
    10: [
        "Manutenção predial solicitada por {area}",
        "Reparo e conservação de instalações - {area}"
    ],
    11: [
        "Honorários profissionais contratados por {area}",
        "Serviço técnico especializado para {area}"
    ],
    12: [
        "Consultoria estratégica para área {area}",
        "Contrato de consultoria externa - {area}"
    ],
    13: [
        "Publicidade institucional - {area}",
        "Propaganda e divulgação para área {area}"
    ],
    14: [
        "Evento corporativo organizado por {area}",
        "Promoção e atividade externa - {area}"
    ],
    15: [
        "Material de marketing solicitado por {area}",
        "Produção de material gráfico - {area}"
    ],
    16: [
        "Abastecimento de veículos - {area}",
        "Combustível e lubrificantes para frota ({area})"
    ],
    17: [
        "Manutenção preventiva de veículos - {area}",
        "Reparo de frota vinculado à área {area}"
    ],
    18: [
        "Passagem aérea para colaborador de {area}",
        "Viagem corporativa - {area}"
    ],
    19: [
        "Hospedagem em viagem a serviço de {area}",
        "Diária de hotel para equipe {area}"
    ],
    20: [
        "Alimentação em viagem corporativa - {area}",
        "Despesa de refeição em deslocamento ({area})"
    ],
    21: [
        "Seguro contratado pela área {area}",
        "Apólice de seguro - {area}"
    ],
    22: [
        "Taxas bancárias incidentes em {area}",
        "Tarifas de serviços financeiros - {area}"
    ],
    23: [
        "Juros sobre operação financeira - {area}",
        "Encargos de empréstimo vinculados a {area}"
    ],
    24: [
        "Despesa jurídica da área {area}",
        "Honorarium advocatício - {area}"
    ],
    25: [
        "Assessoria contábil prestada a {area}",
        "Serviços contábeis externos - {area}"
    ],
    26: [
        "Certificado digital adquirido por {area}",
        "Renovação de certificados digitais - {area}"
    ],
    27: [
        "Licença de software para área {area}",
        "Contrato de SaaS vinculado a {area}"
    ],
    28: [
        "Manutenção de equipamentos de {area}",
        "Reparo de hardware solicitado por {area}"
    ],
    29: [
        "Suporte técnico para sistemas de {area}",
        "Atendimento de TI requisitado por {area}"
    ],
    30: [
        "Treinamento corporativo para equipe {area}",
        "Capacitação profissional vinculada a {area}"
    ],
    31: [
        "Receita de aluguel creditada em {area}",
        "Locação de espaço comercial - {area}"
    ],
    32: [
        "Royalties recebidos por {area}",
        "Direitos de propriedade intelectual - {area}"
    ],
    33: [
        "Comissão recebida por {area}",
        "Receita de comissão comercial - {area}"
    ],
    34: [
        "Resultado de equivalência patrimonial - {area}",
        "Participação nos resultados vinculada a {area}"
    ],
    35: [
        "Venda de ativo imobilizado por {area}",
        "Alienação de bem patrimonial - {area}"
    ],
    36: [
        "Doação recebida em favor de {area}",
        "Recebimento de subvenção - {area}"
    ],
    37: [
        "Receita eventual registrada em {area}",
        "Receita extraordinária - {area}"
    ],
    38: [
        "Desconto concedido em operação de {area}",
        "Abatimento comercial registrado em {area}"
    ],
    39: [
        "Devolução de venda processada por {area}",
        "Estorno de receita - {area}"
    ],
    40: [
        "Imposto sobre venda apurado em {area}",
        "Tributo incidente sobre receita de {area}"
    ]
}

def gerar_descricao(row):
    categoria = row["id_categoria_raw"]
    area = row["codigo_area_raw"]
    template = np.random.choice(templates[categoria])
    return template.format(area=area)

df_base_anomalo["descricao"] = df_base_anomalo.apply(gerar_descricao, axis=1)

df_base_anomalo

In [ ]:
import uuid

df_base_anomalo["data_competencia"] = df_base_anomalo["data_competencia"].dt.to_timestamp()

df_lotes = (
    df_base_anomalo[["data_competencia"]]
    .drop_duplicates()
    .sort_values("data_competencia")
    .reset_index(drop=True)
)

# A7: ingestion_id deterministico. Antes era uuid.uuid4(), entao toda regeracao
# produzia batch ids diferentes -- as 39.792 linhas divergiam entre duas
# execucoes com o mesmo seed, quebrando a reprodutibilidade. uuid5 deriva o id
# do proprio lote (namespace fixo + competencia), garantindo o mesmo UUID
# sempre, sem perder o formato UUID esperado pela coluna CHAR(36).
NAMESPACE_SCAP = uuid.uuid5(uuid.NAMESPACE_DNS, "scap-project.tcc")

df_lotes["ingestion_id"] = [
    str(uuid.uuid5(NAMESPACE_SCAP, f"TRANSACOES_FINANCEIRAS|{c:%Y-%m}"))
    for c in df_lotes["data_competencia"]
]

df_lotes["ingestion_ts"] = (
    df_lotes["data_competencia"] + pd.DateOffset(months=1) + pd.DateOffset(days=4)
)

df_base_anomalo = df_base_anomalo.merge(
    df_lotes,
    on="data_competencia",
    how="left"
)

df_base_anomalo["source_system"] = "ERP_CORPORATIVO"
df_base_anomalo["source_entity"] = "TRANSACOES_FINANCEIRAS"
df_base_anomalo["row_seq"] = range(1, len(df_base_anomalo) + 1)


In [ ]:
import hashlib

colunas_hash = [
    "id_transacao_raw",
    "data_transacao",
    "data_competencia",
    "id_fornecedor_raw",
    "codigo_area_raw",
    "id_categoria_raw",
    "tipo_transacao",
    "valor_bruto",
    "valor_liquido",
    "status_pagamento",
    "moeda",
    "forma_pagamento",
    "descricao"
]

for col in colunas_hash:
    if col not in df_base_anomalo.columns:
        df_base_anomalo[col] = None

print("Colunas antes do hash:", df_base_anomalo.columns.tolist())

# A8: hash estavel. Antes o conteudo vinha de str(valor) direto, entao
# diferencas de ultimo bit em float (observada 3,6e-12, efeito de versao de
# numpy/pandas) mudavam o hash de 175 linhas entre execucoes. Agora os campos
# monetarios e de data sao formatados antes de entrar no hash, o que torna o
# raw_row_hash comparavel entre ambientes -- que e justamente para o que ele
# serve (deteccao de mudanca na origem).
COLUNAS_MONETARIAS = {"valor_bruto", "valor_liquido"}
COLUNAS_DATA = {"data_transacao", "data_competencia"}

def formatar_para_hash(col, valor):
    if pd.isnull(valor):
        return ""
    if col in COLUNAS_MONETARIAS:
        return f"{float(valor):.2f}"
    if col in COLUNAS_DATA:
        return pd.Timestamp(valor).strftime("%Y-%m-%d")
    return str(valor)

def gerar_hash_linha(row):
    texto_concat = "|".join(formatar_para_hash(col, row.get(col, "")) for col in colunas_hash)
    return hashlib.sha256(texto_concat.encode("utf-8")).hexdigest()

df_base_anomalo["raw_row_hash"] = df_base_anomalo.apply(gerar_hash_linha, axis=1)

assert df_base_anomalo["raw_row_hash"].nunique() == len(df_base_anomalo), "hash duplicado"
print(f"Hashes gerados: {df_base_anomalo['raw_row_hash'].nunique():,} unicos")


In [ ]:
colunas_raw = [
    "id_transacao_raw",
    "data_transacao",
    "data_competencia",
    "codigo_area_raw",
    "id_fornecedor_raw",
    "tipo_transacao",
    "valor_bruto",
    "valor_liquido",
    "moeda",
    "descricao",
    "id_categoria_raw",
    "forma_pagamento",
    "status_pagamento",
    "ingestion_id",
    "ingestion_ts",
    "source_system",
    "source_entity",
    "row_seq",
    "raw_row_hash"
]

df_pgtos = df_base_anomalo[colunas_raw].copy()

In [ ]:
# ======================================
# 13. GABARITO DE ANOMALIAS (ground truth)
# ======================================
# O rotulo NAO vai para a camada raw: um ERP real nao sabe dizer o que e
# anomalia. Ele e artefato do experimento controlado e vive separado, para
# ser carregado no schema `ml` e usado so na avaliacao do modelo
# (precision/recall/F1/AUC-ROC). Junta-se ao dado por id_transacao_raw.

COLUNAS_GABARITO = [
    "id_transacao_raw",
    "data_transacao",
    "flag_anomalia",
    "perfil_risco",
    "flag_ajuste",
    "flag_alto_valor",
    "fator_aplicado",
    "valor_original",
    "valor_bruto",
]

df_gabarito = df_base_anomalo.reindex(columns=COLUNAS_GABARITO).copy()

# Linhas normais nao passaram pela injecao: o valor "original" e o proprio valor.
df_gabarito["valor_original"] = df_gabarito["valor_original"].fillna(
    df_gabarito["valor_bruto"]
)
df_gabarito["flag_anomalia"] = df_gabarito["flag_anomalia"].astype(bool)

qtd_anom = int(df_gabarito["flag_anomalia"].sum())
print(f"Gabarito: {len(df_gabarito):,} linhas | anomalias: {qtd_anom:,} "
      f"({qtd_anom / len(df_gabarito):.2%})")
print()
print("Distribuicao por perfil_risco:")
print(df_gabarito["perfil_risco"].value_counts().to_string())
print()
print("Valor bruto medio - normais x anomalas:")
print(df_gabarito.groupby("flag_anomalia")["valor_bruto"].mean().round(2).to_string())

# Sanidade: o gabarito precisa cobrir exatamente as transacoes exportadas.
assert len(df_gabarito) == len(df_pgtos), "gabarito e raw com contagens diferentes"
assert df_gabarito["id_transacao_raw"].is_unique, "id_transacao_raw duplicado"
assert set(df_gabarito["id_transacao_raw"]) == set(df_pgtos["id_transacao_raw"]), \
    "ids do gabarito nao batem com os da camada raw"


In [ ]:
# ======================================
# 14. EXPORTACAO
# ======================================
import os

# A raiz do projeto e localizada subindo a partir do diretorio atual ate achar
# a pasta 'sql' (marcador do repo). Antes era os.path.join(cwd, '..', '..'),
# que so acertava quando o notebook rodava a partir de etl/data_generation/ --
# rodando de generators/ os CSVs iam parar em etl/data/.
def achar_raiz(marcador="sql"):
    caminho = os.path.abspath(os.getcwd())
    while True:
        if os.path.isdir(os.path.join(caminho, marcador)):
            return caminho
        pai = os.path.dirname(caminho)
        if pai == caminho:
            raise RuntimeError(
                f"raiz do projeto nao encontrada (marcador '{marcador}') "
                f"a partir de {os.getcwd()}"
            )
        caminho = pai

workspace_root = achar_raiz()

pasta_destino = os.path.join(workspace_root, "data", "raw", "transacoes_financeiras")
pasta_gabarito = os.path.join(workspace_root, "data", "ground_truth")
os.makedirs(pasta_destino, exist_ok=True)
os.makedirs(pasta_gabarito, exist_ok=True)
print("Working directory:", os.getcwd())
print("Raiz do projeto  :", workspace_root)
print("Export folder    :", pasta_destino)
print("Gabarito folder  :", pasta_gabarito)

df_pgtos["data_competencia"] = df_pgtos["data_competencia"].astype(str)

# 1) Camada raw: um CSV por competencia (sem o rotulo)
total_raw = 0
for competencia, df_mes in df_pgtos.groupby("data_competencia"):
    nome_comp = competencia.replace("-", "_")
    caminho_arquivo = os.path.join(pasta_destino, f"transacoes_{nome_comp}.csv")
    df_mes.to_csv(caminho_arquivo, index=False, sep=",", encoding="utf-8")
    total_raw += len(df_mes)

print()
print(f"Camada raw: {total_raw:,} linhas em "
      f"{df_pgtos['data_competencia'].nunique()} arquivos")

# 2) Gabarito: arquivo unico, carregado depois em ml.transacoes_gabarito
caminho_gabarito = os.path.join(pasta_gabarito, "transacoes_gabarito.csv")
df_gabarito.to_csv(caminho_gabarito, index=False, sep=",", encoding="utf-8")
print(f"Gabarito  : {len(df_gabarito):,} linhas -> {caminho_gabarito}")

print()
print("Exportacao concluida com sucesso.")
